<a href="https://colab.research.google.com/github/0xfffddd/Coding/blob/main/batch_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import zipfile
import os

zip_path = "Transactions_Data.zip"
extract_path = "data"

print(f"📦 start: {zip_path}")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ finished")

📦 start: Transactions_Data.zip
✅ finished


In [10]:
import json

def parse_single_json(file_path):
    rows = []

    with open(file_path, 'r') as f:
        data = json.load(f)

    for transaction in data:
        transaction_id = transaction.get("Transaction_Id")
        date = transaction.get("Created_Date")

        steps = transaction.get("StepsDetails", [])

        for step in steps:
            row = {
                "TransactionID": transaction_id,
                "date": date,
                "functionName": step.get("StepName"),
                "functionStartTime": step.get("StartDate"),
                "EndTime": step.get("EndDate"),
                "Status": "Success" if step.get("Error") in [None, ""] else "Failed"
            }
            rows.append(row)

    return rows

In [11]:
import csv

base_path = "data"

date_folders = sorted(os.listdir(base_path))

total_days = len(date_folders)
print(f"📅 detected {total_days} days of data\n")

for day_index, day_folder in enumerate(date_folders, start=1):

    day_path = os.path.join(base_path, day_folder)

    # 提取日期（Transaction0311 → 311）
    date_str = ''.join(filter(str.isdigit, day_folder))[-3:]

    print(f"\n🚀 start processing Day {day_index}/{total_days}: {day_folder} → output {date_str}.csv")

    stage_path = os.path.join(day_path, "stage_data")

    output_file = f"{date_str}.csv"

    # 写表头
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "TransactionID", "date", "functionName",
            "functionStartTime", "EndTime", "Status"
        ])
        writer.writeheader()

    json_files = [f for f in os.listdir(stage_path) if f.endswith(".json")]
    total_files = len(json_files)

    print(f"📂 this date contains {total_files}  JSON file")

    row_count = 0

    with open(output_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "TransactionID", "date", "functionName",
            "functionStartTime", "EndTime", "Status"
        ])

        for i, file in enumerate(json_files, start=1):
            file_path = os.path.join(stage_path, file)

            try:
                rows = parse_single_json(file_path)
                writer.writerows(rows)
                row_count += len(rows)

                # ✅ 进度提示（核心）
                print(f"   ✅ [{i}/{total_files}] {file} → 累计 {row_count} 行")

            except Exception as e:
                print(f"   ❌ Error: {file} → {e}")

    print(f"🎉 finished {date_str}.csv | rows: {row_count}")

📅 detected 6 days of data


🚀 start processing Day 1/6: Transaction0311 → output 311.csv
📂 this date contains 262  JSON file
   ✅ [1/262] stage_data_0110.json → 累计 9443 行
   ✅ [2/262] stage_data_0223.json → 累计 19526 行
   ✅ [3/262] stage_data_0246.json → 累计 29299 行
   ✅ [4/262] stage_data_0117.json → 累计 39026 行
   ✅ [5/262] stage_data_0219.json → 累计 48158 行
   ✅ [6/262] stage_data_0029.json → 累计 57457 行
   ✅ [7/262] stage_data_0115.json → 累计 67499 行
   ✅ [8/262] stage_data_0215.json → 累计 77583 行
   ✅ [9/262] stage_data_0188.json → 累计 87318 行
   ✅ [10/262] stage_data_0043.json → 累计 96978 行
   ✅ [11/262] stage_data_0040.json → 累计 107049 行
   ✅ [12/262] stage_data_0047.json → 累计 117427 行
   ✅ [13/262] stage_data_0090.json → 累计 126228 行
   ✅ [14/262] stage_data_0092.json → 累计 135857 行
   ✅ [15/262] stage_data_0199.json → 累计 144815 行
   ✅ [16/262] stage_data_0197.json → 累计 154320 行
   ✅ [17/262] stage_data_0218.json → 累计 164282 行
   ✅ [18/262] stage_data_0024.json → 累计 173973 行
   ✅ [19/262]

In [12]:
import pandas as pd

csv_files = [f for f in os.listdir() if f.endswith(".csv") and f != "merged.csv"]

print(f"📊 准备合并 {len(csv_files)} 个 CSV 文件")

df_list = []

for i, file in enumerate(csv_files, start=1):
    print(f"🔄 读取 {i}/{len(csv_files)}: {file}")
    df = pd.read_csv(file)
    df_list.append(df)

print("🧠 开始合并...")
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv("merged.csv", index=False)

print(f"🎉 合并完成！总行数: {len(merged_df)}")

📊 准备合并 6 个 CSV 文件
🔄 读取 1/6: 314.csv
🔄 读取 2/6: 311.csv
🔄 读取 3/6: 313.csv
🔄 读取 4/6: 312.csv
🔄 读取 5/6: 316.csv
🔄 读取 6/6: 315.csv
🧠 开始合并...
🎉 合并完成！总行数: 13820036
